In [0]:
catalog = "cinedata_medallion"
land_schema_name = "landing"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

land_schema = f"{catalog}.{land_schema_name}"
bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path =  f"/Volumes/{catalog}/{land_schema_name}/inputs"

In [0]:
from pyspark.sql.functions import current_timestamp

#Volume path setup
path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_movies_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_movies_info = f"{landing_path}/movies_info_TMDB_IMDB.csv" 
path_movies_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_movies_reviews = f"{landing_path}/movies_reviews.csv"

#Read only
df_credits_and_tags_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path_credits_and_tags)
df_movies_financials_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path_movies_financials)
df_movies_info_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path_movies_info)
df_movies_metrics_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path_movies_metrics)
df_movies_reviews_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path_movies_reviews)

##1. Table creation

In [0]:
#Writing to Bronze with timestamp

#The reason why I chose to use a function instead of creating each one individually is simple:
#Despite loosing flexibility, I get a more robust and easy to maintain code.
#Also, the data in bronze layer is not meant to be changed.

def write_to_bronze(df, table_name, mode="append"):
    (
        df
        .withColumn("ingestion_timestamp", current_timestamp())
        .write
        .format("delta")
        .mode(mode)
        .option("mergeSchema", "true")
        .saveAsTable(f"{bronze_schema}.{table_name}")
    )
    print(f"{bronze_schema}.{table_name} gravada com sucesso.")


MODE = "append" #DONT FORGET TO CHANGE TO "append" BEFORE SHIPPING

write_to_bronze(df_credits_and_tags_raw, "tb_credits_and_tags", MODE)
write_to_bronze(df_movies_financials_raw, "tb_movies_financials", MODE)
write_to_bronze(df_movies_info_raw, "tb_movies_info",MODE)
write_to_bronze(df_movies_metrics_raw, "tb_movies_metrics", MODE)
write_to_bronze(df_movies_reviews_raw, "tb_movies_reviews", MODE)


##2. API Ingestion

In [0]:
#Widget setup

dbutils.widgets.text("data_inicio", "", "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", "", "Data Fim (MM-DD-AAAA)")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

print(f"data_inicio (widget): '{data_inicio}'")
print(f"data_fim (widget): '{data_fim}'")

In [0]:
#It is possible to tweak the values of data_inicio and data_fim using the widgets created previously.
#If not changed, the default values will be used (last 7 days).

from datetime import datetime, timedelta

if not data_inicio or not data_fim:
    today = datetime.today()
    last_seven_days = today - timedelta(days=7)
    data_inicio = last_seven_days.strftime("%m-%d-%Y")  #MM-DD-YYYY
    data_fim = today.strftime("%m-%d-%Y")
    print(f"[FALLBACK] Dates have been calculated: {data_inicio} to {data_fim}")
else:
    print(f"[WIDGET] Dates informed by the user: {data_inicio} to {data_fim}")

In [0]:
import requests

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

print(f"URL built:\n{url}\n")

response = requests.get(url, timeout=30)
response.raise_for_status()

data = response.json()
records = data.get("value", [])
for r in records:
    print(f"  {r}")

In [0]:
#Append to Bronze

from pyspark.sql import Row

if records:
    rows = [Row(**item) for item in records]
    df_cotacao = spark.createDataFrame(rows)

    write_to_bronze(df_cotacao, "tb_cotacao_dolar")
else:
    print("[Warning] The API did not return any record. Nothing to record.")

##3. Validation 

In [0]:
bronze_tables = [
    "tb_credits_and_tags",
    "tb_movies_financials",
    "tb_movies_info",
    "tb_movies_metrics",
    "tb_movies_reviews",
    "tb_cotacao_dolar",
]

for table in bronze_tables:
    print(f"Table: {bronze_schema}.{table}")

    try:
        df = spark.table(f"{bronze_schema}.{table}")
        print(f"Line amount: {df.count()}")
        display(df.limit(5))
    except Exception as e:
        print(f"[ERROR] Table not found: {e}")